In [1]:
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor
from chop import MaseGraph
import chop.passes as passes
from chop.tools import get_trainer
from chop.passes.module import report_trainable_parameters_analysis_pass

# 1. Setup Checkpoints
dataset_name = "cifar10"
checkpoint = "google/vit-base-patch16-224-in21k"


/vol/bitbucket/nr722/adls-project/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/vol/bitbucket/nr722/adls-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from datasets import load_dataset
from transformers import AutoImageProcessor

def get_processed_dataset(dataset_name, checkpoint):
    # 1. Load the raw CIFAR-10 dataset
    raw_datasets = load_dataset(dataset_name)
    
    # Debug: Print column names to be absolutely sure
    print(f"Dataset columns: {raw_datasets['train'].column_names}")
    
    # 2. Load the ImageProcessor (handles resizing to 224x224 and normalization)
    processor = AutoImageProcessor.from_pretrained(checkpoint)
    
    # 3. Define the transformation
    def transform(example_batch):
        # Fallback logic for column names
        img_key = 'img' if 'img' in example_batch else 'image'
        label_key = 'label' if 'label' in example_batch else 'labels'
        
        inputs = processor(example_batch[img_key], return_tensors='pt')
        inputs['labels'] = example_batch[label_key]
        return inputs

    # 4. Apply transformation (using set_transform is faster for images)
    prepared_ds = raw_datasets.with_transform(transform)
    
    return prepared_ds, processor


In [14]:
dataset_dict, processor = get_processed_dataset(dataset_name, checkpoint)

# Use the 'train' and 'test' (or 'validation') splits for the trainer
dataset = dataset_dict["train"]
eval_dataset = dataset_dict["test"] 
# -------------------------

model = AutoModelForImageClassification.from_pretrained(
    checkpoint, 
    num_labels=10,
    ignore_mismatched_sizes=True
)

# Initialize MaseGraph
mg = MaseGraph(
    model,
    hf_input_names=["pixel_values", "labels"],
)

# 4. Run Mase Passes
# Manually provide dummy input to bypass AutoTokenizer error in add_common_metadata_analysis_pass
dummy_input = processor(
    images=[torch.zeros(3, 224, 224)], 
    return_tensors="pt"
)
dummy_input["labels"] = torch.tensor([0])

mg, _ = passes.init_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_input})
mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_input})


Dataset columns: ['img', 'label']


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]],

         [[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]],

         [[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]]])
tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1.,

In [17]:
# 5. Freeze Embeddings (ViT Specific)
# In ViT, the embeddings are under 'vit.embeddings'
for param in mg.model.vit.embeddings.parameters():
    param.requires_grad = False

# Report to verify freezing
_, _ = report_trainable_parameters_analysis_pass(mg.model)

# 6. Training Setup
# Note: get_trainer expects a DatasetDict or similar structure.
# We pass the processed dataset dictionary here.
trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset_dict,  # Pass the DatasetDict
    tokenizer=processor,             # Pass the processor
    evaluate_metric="accuracy",
)
trainer.args.remove_unused_columns = False

from transformers import default_data_collator
trainer.data_collator = default_data_collator

# 7. Execute
eval_results = trainer.evaluate()
print(f"Pre-training accuracy: {eval_results['eval_accuracy']}")

trainer.train()

+------------------------------------------------+------------------------+
| Submodule                                      |   Trainable Parameters |
+================================================+========================+
| vit                                            |               85056000 |
+------------------------------------------------+------------------------+
| vit.embeddings                                 |                      0 |
+------------------------------------------------+------------------------+
| vit.embeddings.patch_embeddings                |                      0 |
+------------------------------------------------+------------------------+
| vit.embeddings.patch_embeddings.projection     |                      0 |
+------------------------------------------------+------------------------+
| vit.embeddings.dropout                         |                      0 |
+------------------------------------------------+------------------------+
| vit.encode

/vol/bitbucket/nr722/adls-project/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Pre-training accuracy: 0.0691


Step,Training Loss
500,0.699700
1000,0.172100
1500,0.138100
2000,0.095600
2500,0.114000
3000,0.093700
3500,0.100400
4000,0.067300
4500,0.084900
5000,0.057600


TrainOutput(global_step=6250, training_loss=0.14230975524902345, metrics={'train_runtime': 1944.0029, 'train_samples_per_second': 25.72, 'train_steps_per_second': 3.215, 'total_flos': 0.0, 'train_loss': 0.14230975524902345, 'epoch': 1.0})

In [18]:
post_eval_results = trainer.evaluate()
print(f"Post-training accuracy: {post_eval_results['eval_accuracy']}")


Post-training accuracy: 0.9849


In [20]:
# 8. Export
mg.export("/vol/bitbucket/nr722/adls-project/vit-cifar10")

INFO     Exporting MaseGraph to /vol/bitbucket/nr722/adls-project/vit-cifar10.pt, /vol/bitbucket/nr722/adls-project/vit-cifar10.mz
INFO     Exporting GraphModule to /vol/bitbucket/nr722/adls-project/vit-cifar10.pt
INFO     Saving full model format
INFO     Exporting MaseMetadata to /vol/bitbucket/nr722/adls-project/vit-cifar10.mz
